In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
import json

# Assuming you have mediapipe solutions initialized in your global scope
mp_holistic = mp.solutions.holistic

In [ ]:
# Prevent OpenCV from competing with TensorFlow's multi-threading,
# which can cause graph execution errors in tf.data.Datasets
cv2.setNumThreads(0)

# A predefined subset of 35 critical facial landmarks (eyes, eyebrows, mouth)
# This reduces the face feature bloat from 1404 values down to 105 values.
SELECTED_FACE_INDICES = [
    # Lips
    61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95,
    # Left Eye & Eyebrow
    33, 133, 159, 145, 46, 52, 53,
    # Right Eye & Eyebrow
    362, 263, 386, 374, 276, 282, 283
]

def extract_keypoints(results):
    """Extracts, normalizes, and flattens landmarks from MediaPipe Holistic results."""

    # POINT B: Define an anchor point for spatial normalization.
    # We use the Pose Nose (landmark 0) if available.
    if results.pose_landmarks:
        anchor_x = results.pose_landmarks.landmark[0].x
        anchor_y = results.pose_landmarks.landmark[0].y
        anchor_z = results.pose_landmarks.landmark[0].z
    else:
        # Fallback if no pose is detected at all
        anchor_x, anchor_y, anchor_z = 0.0, 0.0, 0.0

    # Pose: 33 landmarks. Normalize x, y, z, but keep visibility untouched.
    if results.pose_landmarks:
        pose = np.array([[res.x - anchor_x, res.y - anchor_y, res.z - anchor_z, res.visibility]
                         for res in results.pose_landmarks.landmark]).flatten()
    else:
        pose = np.zeros(33 * 4)

    # Face (POINT A): Only extract the 35 indices defined above and normalize them.
    if results.face_landmarks:
        face = np.array([[results.face_landmarks.landmark[i].x - anchor_x,
                          results.face_landmarks.landmark[i].y - anchor_y,
                          results.face_landmarks.landmark[i].z - anchor_z]
                         for i in SELECTED_FACE_INDICES]).flatten()
    else:
        face = np.zeros(len(SELECTED_FACE_INDICES) * 3)

    # Left Hand: 21 landmarks. Normalize x, y, z.
    if results.left_hand_landmarks:
        lh = np.array([[res.x - anchor_x, res.y - anchor_y, res.z - anchor_z]
                       for res in results.left_hand_landmarks.landmark]).flatten()
    else:
        lh = np.zeros(21 * 3)

    # Right Hand: 21 landmarks. Normalize x, y, z.
    if results.right_hand_landmarks:
        rh = np.array([[res.x - anchor_x, res.y - anchor_y, res.z - anchor_z]
                       for res in results.right_hand_landmarks.landmark]).flatten()
    else:
        rh = np.zeros(21 * 3)

    return np.concatenate([pose, face, lh, rh])

In [ ]:
def precompute_test_dataset(video_dir, precompute_dir, video_ids):
    """
    Step 1: Runs ONCE before evaluation.
    Processes all videos with MediaPipe and saves EVERY frame as fast-loading .npy files.
    Since this is for test data without JSON metadata, it processes the entire video
    from start to finish without boundary frames.
    """
    os.makedirs(precompute_dir, exist_ok=True)
    print(f"Checking/Extracting precomputed test data to {precompute_dir}...")

    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        for i, video_id in enumerate(video_ids):
            npy_path = os.path.join(precompute_dir, f"{video_id}.npy")

            # Skip if already precomputed
            if os.path.exists(npy_path):
                continue

            video_path = os.path.join(video_dir, f"{video_id}.mov")
            if not os.path.exists(video_path):
                print(f"Warning: Video not found at {video_path}")
                continue

            cap = cv2.VideoCapture(video_path)
            raw_frames = []

            # No metadata for f_start, so we start reading from the very beginning.
            try:
                while True:
                    ret, frame = cap.read()

                    # Stop when the video ends natively
                    if not ret:
                        break

                    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    image.flags.writeable = False
                    results = holistic.process(image)

                    # Extract keypoints using your custom function
                    keypoints = extract_keypoints(results)
                    raw_frames.append(keypoints)
            finally:
                cap.release()

            if len(raw_frames) < 2:
                print(f"Warning: Test video {video_id} has less than 2 frames, skipping save.")
                continue

            # Convert the entire raw_frames list into a NumPy array and save it to disk
            all_frames_array = np.array(raw_frames, dtype=np.float32)
            np.save(npy_path, all_frames_array)

            if (i + 1) % 50 == 0:
                print(f"Processed {i + 1}/{len(video_ids)} test videos...")